## Setup

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from torch import from_numpy, Tensor
from torchmetrics import MetricCollection
import torchmetrics
import torchmetrics.text
import evaluate

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from tqdm import tqdm

tqdm.pandas()

In [ ]:
meteor = evaluate.load("meteor")

In [ ]:
df_orig = pd.read_hdf(Path("analysis") / "data" / "all_joined_judged.h5")
df_orig = df_orig[
    df_orig["sample_id"] >= 27000
].copy()  # Only do this for samples from the test split
df_orig

## Compute Metrics

### Binary Tasks

In [ ]:
binary = df_orig[df_orig["answer_type"] == "binary"].copy()
binary.drop(
    columns=["options", "options_sequence", "prediction_parsed_multi"], inplace=True
)
binary["prediction_parsed_binary"].fillna(np.nan, inplace=True)
binary = binary.astype({"answer": int, "prediction_parsed_binary": float}, copy=False)

In [ ]:
def unwrap(res: dict[str, Tensor]) -> dict[str, float]:
    return {k: v.item() for k, v in res.items()}


def compute_binary_metrics(df: pd.DataFrame) -> dict[str, float]:
    metrics = MetricCollection(
        {
            "Accuracy": torchmetrics.Accuracy(task="binary"),
            "Precision": torchmetrics.Precision(task="binary"),
            "Recall": torchmetrics.Recall(task="binary"),
            "F1": torchmetrics.F1Score(task="binary", average="macro"),
        }
    )

    return unwrap(
        metrics(
            from_numpy(df["prediction_parsed_binary"].to_numpy(na_value=-1)),
            from_numpy(df["answer"].to_numpy()),
        )
    )


all_results_binary = (
    binary.groupby(["setting", "model", "mode"])
    .apply(compute_binary_metrics)
    .apply(pd.Series)
    .reset_index()
)
all_results_binary

### Multi Tasks

In [ ]:
multi = df_orig[df_orig["answer_type"] == "multi"].copy()
multi["prediction_parsed_multi"].fillna(np.nan, inplace=True)
multi.drop(columns=["prediction_parsed_binary"], inplace=True)

In [ ]:
def compute_multi_metrics(df: pd.DataFrame) -> dict[str, float]:
    metrics = MetricCollection(
        {
            "Accuracy": torchmetrics.Accuracy(task="multiclass", num_classes=3),
            "Precision": torchmetrics.Precision(
                task="multiclass", average="macro", num_classes=3
            ),
            "Recall": torchmetrics.Recall(
                task="multiclass", average="macro", num_classes=3
            ),
            "F1": torchmetrics.F1Score(
                task="multiclass", num_classes=3, average="macro"
            ),
        }
    )

    ground_truth = df["answer"].to_numpy()

    predictions = (
        df["prediction_parsed_multi"]
        .map({"A": 0, "B": 1, "C": 2, "nan": pd.NA})
        .to_numpy(na_value=-1)
        .astype(np.int32)
    )
    # Make sure we put a wrong answer in case of NaN
    predictions[predictions == -1] = (ground_truth[predictions == -1] + 1) % 3

    return unwrap(
        metrics(
            from_numpy(predictions),
            from_numpy(ground_truth),
        )
    )


all_results_multi = (
    multi.groupby(["setting", "model", "mode"])
    .apply(compute_multi_metrics)
    .apply(pd.Series)
    .reset_index()
)
all_results_multi

### Open Tasks

In [ ]:
open = df_orig[df_orig["answer_type"] == "open"].copy()
open.drop(
    columns=[
        "prediction_parsed_binary",
        "options",
        "answer",
        "options_sequence",
        "prediction_parsed_multi",
    ],
    inplace=True,
)

In [ ]:
def compute_open_metrics(df: pd.DataFrame) -> dict[str, float]:
    inputs = (
        df["prediction_text"].fillna(""),
        df["answer_text"],
    )

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", category=FutureWarning, module="torch.nn.modules.module"
        )

        return {
            "BLEU": torchmetrics.text.BLEUScore()(*inputs).item(),
            "ROUGE": torchmetrics.text.ROUGEScore(
                use_stemmer=True, rouge_keys=("rouge1",)
            )(*inputs)["rouge1_fmeasure"].item(),
            "METEOR": meteor.compute(
                predictions=inputs[0].tolist(),
                references=inputs[1].tolist(),
            )["meteor"],
            "BERTScore": (
                torchmetrics.text.BERTScore(
                    model_name_or_path="roberta-large",
                    lang="en",
                    device="cuda",
                    batch_size=512,
                )(*inputs)["f1"]
                .mean()
                .item()
            ),
            # Also rescale the LLMJudge score to be between 0 and 1
            "LLMJudge": (df["llm_judge_score"].fillna(1).mean() - 1) / 2,
        }


all_results_open = (
    open.groupby(["setting", "model", "mode"])
    .progress_apply(compute_open_metrics)
    .apply(pd.Series)
    .reset_index()
)
all_results_open

## Save / Load metrics

In [ ]:
all_results_binary.to_csv(
    Path("analysis") / "data" / "all_results_binary.csv", index=False
)
all_results_multi.to_csv(
    Path("analysis") / "data" / "all_results_multi.csv", index=False
)
all_results_open.to_csv(Path("analysis") / "data" / "all_results_open.csv", index=False)

In [ ]:
all_results_binary = pd.read_csv(Path("analysis") / "data" / "all_results_binary.csv")
all_results_multi = pd.read_csv(Path("analysis") / "data" / "all_results_multi.csv")
all_results_open = pd.read_csv(Path("analysis") / "data" / "all_results_open.csv")

## Prepare results for paper

In [ ]:
relevant_models = [
    # "Mistral (7B)",
    # "Gemma (2B)",
    "Gemma 7B",
    "Llama 3.1 8B",
    # "Llama 3 (8B)",
    # "Llama 2 (8B)",
    # "Falcon 2 (11B)",
    # "Falcon (7B)",
    "T5 Base",
    # "T5 Small",
    "ChatTS",
]

### Some basic statistics

Mainly about where data was not able to be parsed across variants.

In [ ]:
df_orig

In [ ]:
for_stat = df_orig[df_orig["model"] == "Llama 3.1 (8B)"]
for_stat = for_stat[for_stat["answer_type"].isin(["binary", "multi"])].copy()
for_stat["parsed"] = (
    for_stat["prediction_parsed_multi"].astype(str).replace("nan", pd.NA)
)
for_stat["parsed"].fillna(for_stat["prediction_parsed_binary"], inplace=True)
for_stat.loc[for_stat["parsed"] == "<NA>", "parsed"] = pd.NA
for_stat["parsed"].fillna(pd.NA, inplace=True)  # NaN -> pd.NA
is_open = for_stat["answer_type"] == "open"
for_stat.loc[is_open, "parsed"].fillna(
    for_stat.loc[is_open, "prediction_text"], inplace=True
)

for name, group in for_stat.groupby(["setting", "mode"]):
    print(f"\n{name}")
    print(group["parsed"].value_counts(dropna=False, normalize=True))

### For Section 4.2.1

In [ ]:
setting = "4.2.1"

show_legend = False

variant = "binary"
df = all_results_binary.copy()
human_baseline = 77.89
# variant = "multi"
# df = all_results_multi.copy()
# human_baseline = 86.54
metrics = ["Accuracy", "Precision", "Recall", "F1"]
metric = metrics[3]
ymin, ymax = None, 101
in_percent = True

# variant = "open"
# df = all_results_open.copy()
# human_baseline = 2.2313
# metrics = ["BLEU", "ROUGE", "METEOR", "BERTScore", "LLMJudge"]
# metric = metrics[4]
# ymin, ymax = 1.6, 3.0
# in_percent = False

df["model"] = df["model"].replace(
    {
        "Gemma (7B)": "Gemma 7B",
        "Llama 3.1 (8B)": "Llama 3.1 8B",
    }
)
df = df[df["model"].isin(relevant_models)]
df = df[df["setting"] == setting].copy()
df["mode"] = df["mode"].astype(str)
modes_sorted = sorted(
    [m for m in df["mode"].unique() if m != "full"], key=lambda x: int(x)
) + ["full"]
# Exclude "full" for now
modes_sorted = modes_sorted[:-1]
df_only_full = df[df["mode"] == "full"].copy()
df = df[df["mode"].isin(modes_sorted)].copy()
df["mode"] = df["mode"].astype(int)
# df["mode"] = pd.Categorical(df["mode"], categories=modes_sorted, ordered=True)

sns.set_style("whitegrid")
sns.set_context("talk")

if in_percent:
    df[metric] *= 100
    df_only_full[metric] *= 100

plt.figure(figsize=(8, 6))
grid = sns.relplot(
    data=df.rename(columns={"model": "Model"}),
    x="mode",
    y=metric,
    hue="Model",
    style="Model",
    kind="line",
    markers=True,
    dashes=False,
    hue_order=relevant_models,
    legend="auto" if show_legend else False,
    palette="colorblind",
)

# Add the full + human baselines in the background
plt.axhline(
    y=human_baseline,
    color="gray",
    alpha=0.75,
    linestyle="--",
    label="Humans (Motion)",
    zorder=0,
)
score = df_only_full[df_only_full["model"] == "Llama 3.1 8B"][metric].item()
plt.axhline(
    y=score,
    # use the color from the palette
    color=sns.color_palette("colorblind")[1],
    alpha=0.75,
    linestyle="--",
    label="Llama 3.1 8B (full)",
    zorder=0,
)

# Add a new legend containing the additional lines
if show_legend:
    grid.legend.remove()
    plt.legend(title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.xlabel("Finetuning Dataset Size")
# plt.xlim(0, len(modes_sorted) - 1)
plt.xlim(min(df["mode"]) - 10, max(df["mode"]) + 10)

plt.ylabel(metric)
plt.ylim(ymin, ymax)
if in_percent:
    # Add percentage signs to y-ticks
    plt.gca().set_yticklabels([f"{y.get_text()}%" for y in plt.gca().get_yticklabels()])

plt.grid(True, linestyle="-", alpha=0.5)

sns.despine(top=False, right=False)

plt.savefig(
    Path("analysis") / "plots" / f"{variant}_{metric.lower()}.pdf",
    bbox_inches="tight",
    pad_inches=0,
)
plt.show()

### For Section 4.2.1 - 4.2.4

In [ ]:
df = all_results_binary
df = df[df["model"].isin(["Llama 3.1 (8B)"])]
df = df[(df["setting"] != "4.2.1") | (df["mode"] == "full")]
df = df[df["setting"].isin(["4.2.1", "4.2.2", "4.2.3", "4.2.4"])].copy()
df = df.set_index(["setting", "model", "mode"]) * 100
print(df[["Accuracy", "Precision", "Recall", "F1"]])

In [ ]:
df = all_results_multi
df = df[df["model"].isin(["Llama 3.1 (8B)"])]
df = df[(df["setting"] != "4.2.1") | (df["mode"] == "full")]
df = df[df["setting"].isin(["4.2.1", "4.2.2", "4.2.3", "4.2.4"])].copy()
df = df.set_index(["setting", "model", "mode"]) * 100
print(df[["Accuracy", "Precision", "Recall", "F1"]])

In [ ]:
df = all_results_open
df = df[df["model"].isin(["Llama 3.1 (8B)"])]
df = df[(df["setting"] != "4.2.1") | (df["mode"] == "full")]
df = df[df["setting"].isin(["4.2.1", "4.2.2", "4.2.3", "4.2.4"])].copy()
df = df.set_index(["setting", "model", "mode"])
print(df[["ROUGE", "METEOR", "LLMJudge", "BLEU", "BERTScore"]])

### For Section 4.3 / 4.4

In [ ]:
print(
    (
        all_results_binary[
            all_results_binary["setting"].isin(["4.3", "4.4"])
        ].set_index(["setting", "model", "mode"])
        * 100
    )[["Accuracy", "Precision", "Recall", "F1"]].to_string()
)

In [ ]:
print(
    (
        all_results_multi[all_results_multi["setting"].isin(["4.3", "4.4"])].set_index(
            ["setting", "model", "mode"]
        )
        * 100
    )[["Accuracy", "Precision", "Recall", "F1"]].to_string()
)

In [ ]:
print(
    (
        all_results_open[all_results_open["setting"].isin(["4.3", "4.4"])].set_index(
            ["setting", "model", "mode"]
        )
    )[["ROUGE", "METEOR", "LLMJudge", "BLEU", "BERTScore"]].to_string()
)

## Per-Question-Type Breakdown: Humans vs. xQA-Qwen (AE)

Tables 2 and 3 report Humans and the xQA systems as one aggregate number per answer format. This
section breaks both down over all 45 question types of QuAnTS, laid out like Figure 5.

Needs the cells above that build the `binary`, `multi`, and `open` frames, but *not* the expensive
`compute_open_metrics` cell -- no BERTScore and no GPU are required here.

In [ ]:
# Work no matter whether the notebook is executed from the repo root or from analysis/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
DATA = ROOT / "analysis" / "data"
PLOTS = ROOT / "analysis" / "plots"
PLOTS.mkdir(exist_ok=True)

sns.set_style("whitegrid")
sns.set_context("talk", font_scale=1.15)  # readable once the figure is scaled down in the paper
PALETTE = sns.color_palette("colorblind")
plt.rcParams["pdf.fonttype"] = 42  # TrueType instead of Type 3, required by most camera-ready checks
plt.rcParams["ps.fonttype"] = 42

RNG = np.random.default_rng(0)
N_BOOT = 2_000

# The stored model names predate the paper's renaming: "xQA-Qwen" is the xLSTM-based system, not
# the "NeSy Qwen3 8B" one. The consistency check below pins this mapping to Tables 2 and 3.
SYSTEMS = {
    "Humans": ("4.3", "Human", "organic"),
    "xQA-Qwen on AE": ("4.4", "NeSy xLSTM + Qwen3 (8B) AE", "full"),
}

SPLITS = {"binary": binary, "multi": multi, "open": open}
METRIC_LABELS = {"binary": "F1", "multi": "Macro F1", "open": "LLM judge"}
ANSWER_COLORS = dict(zip(SPLITS, PALETTE))  # binary blue, multi orange, open green, as in Figure 5

In [ ]:
def f1_binary(predictions: np.ndarray, targets: np.ndarray) -> float:
    """Positive-class F1, matching `F1Score(task="binary")`."""
    positive = predictions == 1
    tp = np.count_nonzero(positive & (targets == 1))
    fp = np.count_nonzero(positive & (targets == 0))
    fn = np.count_nonzero(~positive & (targets == 1))
    denominator = 2 * tp + fp + fn
    return 2 * tp / denominator if denominator else 0.0


def f1_macro_multi(predictions: np.ndarray, targets: np.ndarray) -> float:
    """Macro-averaged F1 over the three answer options."""
    scores = []
    for c in range(3):
        predicted = predictions == c
        actual = targets == c
        tp = np.count_nonzero(predicted & actual)
        fp = np.count_nonzero(predicted & ~actual)
        fn = np.count_nonzero(~predicted & actual)
        denominator = 2 * tp + fp + fn
        scores.append(2 * tp / denominator if denominator else 0.0)
    return float(np.mean(scores))


def rows_of(split: pd.DataFrame, system: str) -> pd.DataFrame:
    setting, model, mode = SYSTEMS[system]
    return split[
        (split["setting"] == setting)
        & (split["model"] == model)
        & (split["mode"].astype(str) == mode)
    ]


# One frame per split holding only the two systems, plus the per-response arrays each metric needs.
# Keeping the metrics index-based is what makes 90 groups x 2000 resamples cheap.
COMPARED = {
    name: pd.concat([rows_of(split, s) for s in SYSTEMS]) for name, split in SPLITS.items()
}

binary_predictions = (
    COMPARED["binary"]["prediction_parsed_binary"].to_numpy(na_value=-1) == 1
).astype(np.int64)
binary_targets = COMPARED["binary"]["answer"].to_numpy().astype(np.int64)

multi_targets = COMPARED["multi"]["answer"].to_numpy().astype(np.int64)
multi_predictions = (
    COMPARED["multi"]["prediction_parsed_multi"]
    .map({"A": 0, "B": 1, "C": 2, "nan": pd.NA})
    .to_numpy(na_value=-1)
    .astype(np.int64)
)
# Make sure we put a wrong answer in case of NaN
multi_predictions[multi_predictions == -1] = (multi_targets[multi_predictions == -1] + 1) % 3

open_scores = COMPARED["open"]["llm_judge_score"].fillna(1).to_numpy(dtype=float)

METRICS = {
    "binary": lambda idx: f1_binary(binary_predictions[idx], binary_targets[idx]),
    "multi": lambda idx: f1_macro_multi(multi_predictions[idx], multi_targets[idx]),
    "open": lambda idx: (open_scores[idx].mean() - 1) / 2,
}

# Row positions within each COMPARED frame, per system
POSITIONS = {
    name: {
        system: np.flatnonzero(
            (frame["setting"] == SYSTEMS[system][0])
            & (frame["model"] == SYSTEMS[system][1])
            & (frame["mode"].astype(str) == SYSTEMS[system][2])
        )
        for system in SYSTEMS
    }
    for name, frame in COMPARED.items()
}

In [ ]:
# Pooled over all question types, both systems have to land on the published numbers again --
# this is what keeps the "NeSy xLSTM + Qwen3 (8B) AE" -> "xQA-Qwen on AE" mapping honest.
published = {
    name: pd.read_csv(DATA / f"all_results_{name}.csv").set_index(["setting", "model", "mode"])
    for name in SPLITS
}

for name in SPLITS:
    column = "LLMJudge" if name == "open" else "F1"
    for system, key in SYSTEMS.items():
        value = METRICS[name](POSITIONS[name][system])
        reference = published[name].loc[key, column]
        print(f"{name:>6} / {system:<15}: {value:.6f} (published {reference:.6f})")
        assert np.isclose(value, reference, atol=1e-6), (name, system)

In [ ]:
def score_with_ci(metric, idx: np.ndarray) -> tuple[float, float, float]:
    """The metric on this group of responses, with a percentile bootstrap interval."""
    resampled = np.array(
        [metric(RNG.choice(idx, size=idx.size, replace=True)) for _ in range(N_BOOT)]
    )
    low, high = np.percentile(resampled, [2.5, 97.5])
    return metric(idx), low, high


records = []
for name, frame in COMPARED.items():
    question_types = frame["question_type"].to_numpy()
    for system in SYSTEMS:
        available = POSITIONS[name][system]
        for question_type in sorted(np.unique(question_types)):
            idx = available[question_types[available] == question_type]
            assert idx.size, (name, system, question_type)
            score, low, high = score_with_ci(METRICS[name], idx)
            records.append(
                {
                    "system": system,
                    "answer_type": name,
                    "question_type": question_type,
                    "n": idx.size,
                    "score": score,
                    "ci_low": low,
                    "ci_high": high,
                }
            )

breakdown = pd.DataFrame(records)
assert breakdown["question_type"].nunique() == 45, breakdown["question_type"].nunique()
assert len(breakdown) == 45 * len(SYSTEMS), len(breakdown)
breakdown.to_csv(DATA / "question_type_breakdown.csv", index=False)
breakdown.head()

In [ ]:
# binary -> multi -> open, alphabetically within each block. That is Figure 5's order read
# bottom-up; this figure reads top-down instead.
BLOCKS = {
    name: sorted(breakdown.loc[breakdown["answer_type"] == name, "question_type"].unique())
    for name in SPLITS
}
QUESTION_TYPE_ORDER = [q for name in SPLITS for q in BLOCKS[name]]
assert len(QUESTION_TYPE_ORDER) == 45, len(QUESTION_TYPE_ORDER)
{name: len(types) for name, types in BLOCKS.items()}

In [ ]:
from matplotlib.transforms import blended_transform_factory

OFFSETS = {"Humans": -0.2, "xQA-Qwen on AE": 0.2}  # first system on top of each pair


def shade(colour, system: str):
    """Humans get a washed-out, hatched version of the answer type's colour, xQA-Qwen the solid one.

    Colour therefore stays free to encode the answer type, as in Figure 5, and the hatch keeps the
    two systems apart even when the figure is printed in grayscale.
    """
    if system == "Humans":
        return tuple(1 - 0.45 * (1 - c) for c in colour)
    return colour


# Row positions. A gap between the answer-type blocks is what sets them apart, together with
# their colour -- no separator rules needed.
GAP = 0.7
ROWS, cursor = {}, 0.0
for i, name in enumerate(SPLITS):
    if i:
        cursor += GAP
    for question_type in BLOCKS[name]:
        ROWS[question_type] = cursor
        cursor += 1

fig, ax = plt.subplots(figsize=(17, 20))
indexed = breakdown.set_index(["system", "question_type"])

for system in SYSTEMS:
    subset = indexed.loc[system].loc[QUESTION_TYPE_ORDER]
    values = subset["score"].to_numpy() * 100
    errors = np.abs(np.stack([subset["ci_low"], subset["ci_high"]]) * 100 - values)
    y = np.array([ROWS[q] for q in QUESTION_TYPE_ORDER]) + OFFSETS[system]
    colours = [ANSWER_COLORS[a] for a in subset["answer_type"]]

    ax.barh(
        y,
        values,
        height=0.38,
        color=[shade(c, system) for c in colours],
        edgecolor=colours,
        linewidth=0.8,
        hatch="//" if system == "Humans" else None,
        zorder=2,
    )
    ax.errorbar(
        values,
        y,
        xerr=errors,
        fmt="none",
        ecolor="0.25",
        elinewidth=1.2,
        capsize=3,
        zorder=3,
    )

ax.set_yticks([ROWS[q] for q in QUESTION_TYPE_ORDER], QUESTION_TYPE_ORDER)
ax.set_ylim(cursor - 0.5, -0.5)
ax.set_xlim(0, 103)
ax.set_xticks(np.arange(0, 101, 20), [f"{x}%" for x in range(0, 101, 20)])
ax.set_xlabel("Score")  # the legend already spells out the metric per answer type
ax.grid(True, axis="x", linestyle="-", alpha=0.5)
ax.grid(False, axis="y")
# Keep only the left spine, next to the question-type labels
sns.despine(ax=ax, top=True, right=True, bottom=True, left=False)

# Colour already says which answer type, so each block is named down the right-hand side rather
# than in a legend of its own -- that keeps the figure from growing a wide second header row.
block_anchor = blended_transform_factory(ax.transAxes, ax.transData)
block_labels = [
    ax.text(
        1.015,
        (ROWS[BLOCKS[name][0]] + ROWS[BLOCKS[name][-1]]) / 2,
        f"{name.capitalize()} ({METRIC_LABELS[name]})",
        transform=block_anchor,
        rotation=270,
        ha="left",
        va="center",
        color=ANSWER_COLORS[name],
    )
    for name in SPLITS
]

# Only the shading that tells the two systems apart still needs a legend. It is centred over the
# bars, which leaves the space above the question-type labels free for their heading. The vertical
# offset is in axes fractions, and this axes is tall, so it is small.
system_legend = ax.legend(
    handles=[
        plt.Rectangle(
            (0, 0),
            1,
            1,
            facecolor=shade((0.35, 0.35, 0.35), system),
            edgecolor="0.35",
            hatch="//" if system == "Humans" else None,
            label=system,
        )
        for system in SYSTEMS
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, 1.002),
    ncol=len(SYSTEMS),
    fontsize="small",
    frameon=False,
)

fig.tight_layout()

# Heading for the question-type column, above its labels rather than rotated beside them, and
# vertically centred on the legend row next to it -- which takes a draw pass to measure
fig.canvas.draw()
legend_box = system_legend.get_window_extent()
heading_offset = (
    (legend_box.y0 + legend_box.height / 2 - ax.transAxes.transform((0, 1))[1]) / fig.dpi * 72
)
heading = ax.annotate(
    "Question Type",
    xy=(0, 1),
    xycoords="axes fraction",
    xytext=(-10, heading_offset),
    textcoords="offset points",
    ha="right",
    va="center",
)

# The legend, the heading, and the block labels sit outside the axes, so they have to be named
# explicitly or the tight bounding box crops them away
fig.savefig(
    PLOTS / "question_type_breakdown.pdf",
    bbox_inches="tight",
    pad_inches=0,
    bbox_extra_artists=[system_legend, heading, *block_labels],
)
plt.show()

In [ ]:
# A technical caption for the figure, with every number read back off the computed frame so that
# it cannot drift from what was actually plotted.
#
# The human count is taken over distinct (sample_id, question) pairs rather than over rows: in four
# cases a clip carries the same question text under two question_ids, so the one human answer joins
# onto both. That gives 840 + 820 + 440 = 2100 responses, matching the n reported in Tables 2 and
# 3, against 2104 rows. The published scores are computed over the rows, and the difference is
# negligible (binary F1 77.89 vs 77.74, open LLM judge 0.6156 vs 0.6148, multi unchanged), so the
# rows are what is scored here too.
human_responses = sum(
    len(rows_of(split, "Humans").drop_duplicates(["sample_id", "question"]))
    for split in SPLITS.values()
)
totals = breakdown.groupby("system")["n"].sum()
spans = breakdown.groupby("system")["n"].agg(["min", "max"])

print(
    f"Per-question-type performance of Humans (n = {human_responses:,} responses) and xQA-Qwen on "
    f"AE (n = {totals['xQA-Qwen on AE']:,} answers) on the QuAnTS test split, over all "
    f"{len(QUESTION_TYPE_ORDER)} question types, ordered binary -> multi -> open and "
    "alphabetically within each block. Bars show positive-class F1 (binary), macro-averaged F1 "
    "over the three options (multi), and the LLM-judge score rescaled from [1, 3] to [0, 1] "
    "(open); answers that cannot be parsed are counted as wrong, as everywhere else in the paper. "
    f"Error bars are 95% percentile bootstrap intervals from {N_BOOT:,} resamples of the responses "
    "within each question type (seed 0); they are not corrected for multiple comparisons across "
    f"the {len(QUESTION_TYPE_ORDER)} types. They are consistently wider for Humans because the "
    f"study contributes only {spans.loc['Humans', 'min']}-{spans.loc['Humans', 'max']} responses "
    f"per question type, against {spans.loc['xQA-Qwen on AE', 'min']}-"
    f"{spans.loc['xQA-Qwen on AE', 'max']} answers for xQA-Qwen on AE."
)